In [1]:
import os
import datetime
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
import joblib

BATCH_SIZE = 1024
EPOCHS     = 50

# ─────────────────────────────────────────────────────────────
# 1. LOAD & VERIFY
# ─────────────────────────────────────────────────────────────
print("Working directory:", os.getcwd())
print("combined_output.csv exists:", os.path.exists("combined_output.csv"))
t = os.path.getmtime("combined_output.csv")
print("Last modified:", datetime.datetime.fromtimestamp(t))

dataset = pd.read_csv('combined_output.csv')
dataset = dataset.dropna()

print("\nRaw label counts:")
print(dataset["Label"].value_counts().to_string())

# ─────────────────────────────────────────────────────────────
# 2. LABEL MAPPING
# ─────────────────────────────────────────────────────────────
def map_to_category(label):
    label = str(label).upper().strip()
    if label == "BENIGN":
        return "Benign"
    elif label.startswith("DDOS-"):
        return "DDoS"
    elif label.startswith("DOS-"):
        return "DoS"
    elif label.startswith("MIRAI-") or label == "BACKDOOR_MALWARE":
        return "Mirai"
    elif label.startswith("RECON-") or label == "VULNERABILITYSCAN":
        return "Reconnaissance"
    elif label in ["DNS_SPOOFING", "MITM-ARPSPOOFING"]:
        return "Spoofing"
    elif label == "DICTIONARYBRUTEFORCE":
        return "Brute_Force"
    elif label in ["SQLINJECTION", "XSS", "COMMANDINJECTION",
                   "UPLOADING_ATTACK", "BROWSERHIJACKING"]:
        return "Web_Attack"
    else:
        return "Unknown"

dataset["category"] = dataset["Label"].apply(map_to_category)

unknown_count = (dataset["category"] == "Unknown").sum()
if unknown_count > 0:
    print(f"\n⚠️  Dropping {unknown_count:,} rows with unmapped labels")
    dataset = dataset[dataset["category"] != "Unknown"]

print(f"\nColumns: {len(dataset.columns)}")
print("\nCategory distribution after mapping:")
print(dataset["category"].value_counts().to_string())

# ─────────────────────────────────────────────────────────────
# 3. LABEL ENCODING
# ─────────────────────────────────────────────────────────────
le = LabelEncoder()
dataset['category'] = le.fit_transform(dataset['category'])
print("\nLabelling done")
print("Classes:", list(le.classes_))

for cls in ["Benign", "Brute_Force", "DDoS", "DoS", "Mirai",
            "Reconnaissance", "Spoofing", "Web_Attack"]:
    idx   = le.transform([cls])[0]
    count = (dataset['category'] == idx).sum()
    print(f"  {'✅' if count > 0 else '❌'} {cls}: {count:,} samples")

# ─────────────────────────────────────────────────────────────
# 4. TRAIN / VALIDATION / TEST SPLIT
#    Validation carved out BEFORE resampling → real distribution
# ─────────────────────────────────────────────────────────────
X = dataset.drop(columns=["Label", "category"])
y = dataset["category"]

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.1, random_state=42, stratify=y_trainval
)

print(f"\nSplit sizes — Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}")

# ─────────────────────────────────────────────────────────────
# 5. CLEAN INFINITIES & IMPUTE
# ─────────────────────────────────────────────────────────────
X_train = X_train.copy()
X_val   = X_val.copy()
X_test  = X_test.copy()

X_train[np.isinf(X_train)] = np.nan
X_val[np.isinf(X_val)]     = np.nan
X_test[np.isinf(X_test)]   = np.nan

imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_val   = imputer.transform(X_val)
X_test  = imputer.transform(X_test)
print("Data split done")

# ─────────────────────────────────────────────────────────────
# 6. SCALE
# ─────────────────────────────────────────────────────────────
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)
print("Data scaling done")

# ─────────────────────────────────────────────────────────────
# 7. STEP 1 — UNDERSAMPLE dominant classes on TRAIN ONLY
# ─────────────────────────────────────────────────────────────
ddos_idx  = le.transform(['DDoS'])[0]
dos_idx   = le.transform(['DoS'])[0]
brute_idx = le.transform(['Brute_Force'])[0]
web_idx   = le.transform(['Web_Attack'])[0]
spoof_idx = le.transform(['Spoofing'])[0]
benign_idx= le.transform(['Benign'])[0]
recon_idx = le.transform(['Reconnaissance'])[0]

train_counts = pd.Series(y_train).value_counts()
print("\nTraining counts before undersampling:")
print(train_counts.rename(index=dict(enumerate(le.classes_))).to_string())

under_strategy = {}
if train_counts.get(ddos_idx, 0) > 500000:
    under_strategy[ddos_idx] = 500000
if train_counts.get(dos_idx, 0) > 300000:
    under_strategy[dos_idx]  = 300000

if under_strategy:
    rus = RandomUnderSampler(sampling_strategy=under_strategy, random_state=42)
    X_train, y_train = rus.fit_resample(X_train, y_train)
    print("\nAfter undersampling:")
    print(pd.Series(y_train).value_counts()
          .rename(index=dict(enumerate(le.classes_))).to_string())

# ─────────────────────────────────────────────────────────────
# 8. STEP 2 — OVERSAMPLE Brute_Force and Web_Attack only
#    Spoofing has ~55K real samples — leave it untouched
# ─────────────────────────────────────────────────────────────
train_counts = pd.Series(y_train).value_counts()
TARGET       = 30000

over_strategy = {}
for idx in [brute_idx, web_idx]:
    current = train_counts.get(idx, 0)
    if current == 0:
        print(f"\n❌ '{le.classes_[idx]}' has 0 samples — regenerate combined_output.csv")
    elif current < TARGET:
        over_strategy[idx] = TARGET

if over_strategy:
    ros = RandomOverSampler(sampling_strategy=over_strategy, random_state=42)
    X_train, y_train = ros.fit_resample(X_train, y_train)
    print("\nAfter oversampling:")
    print(pd.Series(y_train).value_counts()
          .rename(index=dict(enumerate(le.classes_))).to_string())

print("\nFinal training distribution:")
print(pd.Series(y_train).value_counts()
      .rename(index=dict(enumerate(le.classes_))).to_string())

# ─────────────────────────────────────────────────────────────
# 9. CLASS WEIGHTS — best performing settings restored
#    From the run that gave: Benign 0.69, Recon 0.63, Spoofing 0.47
#    Majority classes at 1.0 to preserve training stability
# ─────────────────────────────────────────────────────────────
num_classes = len(np.unique(y_train))
class_weights = {i: 1.0 for i in range(num_classes)}

class_weights[benign_idx] = 4.0   # stable — keep
class_weights[recon_idx]  = 5.0   # stable — keep
class_weights[spoof_idx]  = 7.0   # restored — best run gave F1 0.47
class_weights[brute_idx]  = 8.0   # restored — best run gave F1 0.19
class_weights[web_idx]    = 8.0   # restored — best run gave F1 0.06

print("\nFinal class weights:")
for i, cls in enumerate(le.classes_):
    print(f"  {cls}: {class_weights[i]:.2f}")

# ─────────────────────────────────────────────────────────────
# 10. MODEL
# ─────────────────────────────────────────────────────────────
input_dim = X_train.shape[1]
print(f"\nInput dimension: {input_dim}")
print(f"Number of classes: {num_classes}")

model = tf.keras.Sequential([
    tf.keras.Input(shape=(input_dim,)),

    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(32, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

# Stable LR that gave 20 epochs of training
optimizer = Adam(learning_rate=5e-5, clipnorm=1.0)  # halved — weighted loss ~2.8 needs smaller steps
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    )
]

# ─────────────────────────────────────────────────────────────
# 11. TRAIN
# ─────────────────────────────────────────────────────────────
history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    class_weight=class_weights,
    callbacks=callbacks
)

# ─────────────────────────────────────────────────────────────
# 12. EVALUATE
# ─────────────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"\nTest accuracy: {test_acc:.4f}")

y_pred         = model.predict(X_test)
y_pred_classes = y_pred.argmax(axis=1)

report = classification_report(
    y_test, y_pred_classes,
    target_names=le.classes_,
    zero_division=0
)
print(report)

# ─────────────────────────────────────────────────────────────
# 13. SAVE ARTIFACTS
# ─────────────────────────────────────────────────────────────
model.save('baseline_model.keras')
joblib.dump(scaler,  'scaler.pkl')
joblib.dump(imputer, 'imputer.pkl')
joblib.dump(le,      'label_encoder.pkl')
np.save('X_test.npy', X_test)
np.save('y_test.npy', y_test)
print("\nAll Phase 1 artifacts saved.")

Working directory: /Users/yasuke/Desktop/UNB2026/NetworkSecurity/Project/CS6415
combined_output.csv exists: True
Last modified: 2026-03-19 01:58:07.496686

Raw label counts:
Label
DDOS-ICMP_FLOOD            1519937
DDOS-UDP_FLOOD             1142048
DDOS-TCP_FLOOD              951670
DDOS-PSHACK_FLOOD           864976
DDOS-SYN_FLOOD              856583
DDOS-RSTFINFLOOD            855285
DDOS-SYNONYMOUSIP_FLOOD     759110
DOS-UDP_FLOOD               700617
DOS-TCP_FLOOD               563776
DOS-SYN_FLOOD               427640
BENIGN                      231656
MIRAI-GREETH_FLOOD          209508
MIRAI-UDPPLAIN              187662
MIRAI-GREIP_FLOOD           158428
DDOS-ICMP_FRAGMENTATION      95553
VULNERABILITYSCAN            78775
MITM-ARPSPOOFING             64855
DDOS-UDP_FRAGMENTATION       60548
DDOS-ACK_FRAGMENTATION       60211
DNS_SPOOFING                 37629
RECON-HOSTDISCOVERY          28444
RECON-OSSCAN                 20764
RECON-PORTSCAN               17105
DOS-HTTP_FLOOD 

2026-03-19 17:17:39.921047: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5
2026-03-19 17:17:39.922244: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-03-19 17:17:39.922248: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-03-19 17:17:39.923346: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-19 17:17:39.923372: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │        10,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 8)              │           264 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 55,528 (216.91 KB)

 Trainable params: 54,632 (213.41 KB)

 Non-trainable params: 896 (3.50 KB)

Epoch 1/50


2026-03-19 17:17:40.983488: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1387/1387 ━━━━━━━━━━━━━━━━━━━━ 27s 18ms/step - accuracy: 0.5711 - loss: 3.4340 - val_accuracy: 0.7360 - val_loss: 0.6046 - learning_rate: 5.0000e-05
Epoch 2/50
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.6708 - loss: 2.4721 - val_accuracy: 0.7330 - val_loss: 0.5315 - learning_rate: 5.0000e-05
Epoch 3/50
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 24s 17ms/step - accuracy: 0.6864 - loss: 2.2644 - val_accuracy: 0.7303 - val_loss: 0.5121 - learning_rate: 5.0000e-05
Epoch 4/50
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 24s 18ms/step - accuracy: 0.6942 - loss: 2.1892 - val_accuracy: 0.7338 - val_loss: 0.5069 - learning_rate: 5.0000e-05
Epoch 5/50
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 24s 18ms/step - accuracy: 0.6963 - loss: 2.1823 - val_accuracy: 0.7242 - val_loss: 0.5179 - learning_rate: 5.0000e-05
Epoch 6/50
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 24s 18ms/step - accuracy: 0.6936 - loss: 2.2024 - val_accuracy: 0.7146 - val_loss: 0.5342 - learning_rate: 5.0000e-05
Epoch 7/50
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/